# Darcy equation: exercise 2

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$k=I$ the matrix permeability and $g=(0, 1)$ a vector source term, we want to solve the following problem: find $({q}, p)$ such that
$$
\left\{
\begin{array}{ll}
\begin{array}{l} 
k^{-1} {q} + \nabla p = {g}\\
\nabla \cdot {q} = 0
\end{array}
&\text{in } \Omega
\end{array}
\right.
$$
with boundary conditions:
$$ p = 1 \text{ on } \partial_{top} \Omega \qquad p = 0 \text{ on } \partial_{bottom} \Omega \qquad \nu \cdot q = 0 \text{ on } \partial_{left} \Omega \cup \partial_{right} \Omega$$

This is the guided ("fill in the code") version of `ex2.ipynb` -- work through the cells in order, completing each `# TODO`. Compare against `ex2.ipynb` once you're done, or if you get stuck.

First we import some of the standard modules.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We create now the grid, in this example we consider a 2-dimensional grid.

In [ ]:
mesh_size = 0.1
# creation of the grid
sd = pg.unit_grid(2, mesh_size, as_mdg=False)
# compute the geometrical properties of the grid
sd.compute_geometry()

Let us declare the finite element spaces that we are going to use

In [ ]:
key = "flow"

# declare the discretization objects, useful to setup the data
rt0 = pg.RT0(key)
p0 = pg.PwConstants(key)

# build the degrees of freedom
dofs = np.array([rt0.ndof(sd), p0.ndof(sd)])

With the following code we set the data, in particular the permeability tensor and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

In [ ]:
# TODO: unitary permeability tensor
inv_perm = __TODO__
param = {pg.SECOND_ORDER_TENSOR: inv_perm}
data = pp.initialize_data({}, key, param)

# with the following steps we identify the portions of the boundary
# to impose the boundary conditions
left = np.isclose(sd.face_centers[0, :], 0)
right = np.isclose(sd.face_centers[0, :], 1)
left_right = np.logical_or(left, right)

bottom = np.isclose(sd.face_centers[1, :], 0)
top = np.isclose(sd.face_centers[1, :], 1)
bottom_top = np.logical_or(bottom, top)

ess_p = np.zeros(dofs[1], dtype=bool)


# compute the pressure boundary condition, which is a natural condition for the RT0 space
def p_bc(x):
    return x[1]


# TODO: assemble the natural (pressure) boundary condition on bottom_top
bc_val = __TODO__
bc_ess = np.hstack((left_right, ess_p))


# TODO: compute the source term being a buoyancy pointing upward, by
# interpolating the vector [0, 1, 0] with rt0.interpolate and multiplying
# by the RT0 mass matrix
mass = rt0.assemble_mass_matrix(sd)
vector_source = __TODO__

Once the data are assigned to the grid, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
A & -B^\top\\
B & 0
\end{array}
\right)
\left(
\begin{array}{c} 
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
p_{\partial} + g\\ 
0
\end{array}
\right)
$$<br>
where $p_{\partial}$ is the vector associated to the pressure boundary conditions. Once the matrix is created, we also construct the right-hand side containing the boundary conditions.

In [ ]:
# construct the local matrices
A = rt0.assemble_mass_matrix(sd, data)
mass_p0 = p0.assemble_mass_matrix(sd)
B = mass_p0 @ rt0.assemble_diff_matrix(sd)

# TODO: assemble the saddle point problem with sps.block_array
spp = __TODO__

# TODO: assemble the right-hand side (add the boundary term and vector
# source to the q-block, i.e. the first dofs[0] entries)
rhs = np.zeros(dofs.sum())
rhs[: dofs[0]] += __TODO__

We need to solve the linear system, PyGeoN provides a framework for that. The actual imposition of essential boundary conditions (flux boundary conditions) might change the symmetry of the global system, the class `pg.LinearSystem` preserves this structure by internally eliminating these degrees of freedom. Once the problem is solved, we extract the two solutions $q$ and $p$.

In [ ]:
# solve the problem
ls = pg.LinearSystem(spp, rhs)
ls.flag_ess_bc(bc_ess, np.zeros(dofs.sum()))
x = ls.solve()

# TODO: split the solution into the components q and p
idx = np.cumsum(dofs[:-1])
q, p = __TODO__

Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the flux in each cell center as vector. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# post process variables
proj_q = rt0.eval_at_cell_centers(sd)
# TODO: project q to cell centers and reshape to (pg.AMBIENT_DIM, -1)
cell_q = __TODO__
cell_p = p0.eval_at_cell_centers(sd) @ p

save = pp.Exporter(sd, "sol", folder_name="ex2")
save.write_vtu([("cell_p", cell_p), ("cell_q", cell_q)])

In [ ]:
# Consistency check -- once your implementation is correct, this should pass
assert np.isclose(np.linalg.norm(cell_p), 9.11100242566703)
assert np.isclose(np.linalg.norm(cell_q), 0)